# Dataset exploration and read-only audit

This notebook performs a reproducible, **read-only** audit of the raw tweet and network-edge files. It does not create a sampled dataset, assign semantics to `Target`, or build a diffusion model.

## 1. Dataset loading

All paths are resolved relative to the project root. Outputs are written only to `results/figures/`; the contents of `data/raw/` are never modified.

In [ ]:
from pathlib import Path
from collections import Counter
import json
import math
import re

import pandas as pd
from PIL import Image, ImageDraw, ImageFont

# This makes the notebook portable whether it is started from the project root or notebooks/.
candidates = [Path.cwd().resolve(), Path.cwd().resolve().parent]
PROJECT_ROOT = next((p for p in candidates if (p / 'data' / 'raw' / 'master_tweets.csv').exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Run this notebook from the project root or its notebooks directory.')

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
FIGURES_DIR = PROJECT_ROOT / 'results' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

TWEETS_PATH = RAW_DIR / 'master_tweets.csv'
EDGES_PATH = RAW_DIR / 'network_edges.csv'
tweets = pd.read_csv(TWEETS_PATH)
edges = pd.read_csv(EDGES_PATH)
print(f'Loaded {TWEETS_PATH.relative_to(PROJECT_ROOT)} and {EDGES_PATH.relative_to(PROJECT_ROOT)}')

# Lightweight Pillow charts keep this audit runnable without requiring a plotting backend.
def save_bar_chart(labels, values, title, filename, color=(55, 126, 184)):
    width, height, margin = 1200, 680, 85
    image = Image.new('RGB', (width, height), 'white')
    draw = ImageDraw.Draw(image)
    font = ImageFont.load_default()
    max_value = max(values) if values else 1
    plot_width, plot_height = width - 2 * margin, height - 2 * margin
    draw.text((margin, 25), title, fill='black', font=font)
    draw.line((margin, height-margin, width-margin, height-margin), fill='black', width=2)
    draw.line((margin, margin, margin, height-margin), fill='black', width=2)
    bar_space = plot_width / max(len(values), 1)
    bar_width = max(2, int(bar_space * 0.72))
    for idx, (label, value) in enumerate(zip(labels, values)):
        x0 = int(margin + idx * bar_space + (bar_space - bar_width) / 2)
        x1 = x0 + bar_width
        y1 = height - margin
        y0 = int(y1 - (value / max_value) * (plot_height - 45)) if max_value else y1
        draw.rectangle((x0, y0, x1, y1), fill=color)
        draw.text((x0, max(margin + 22, y0 - 15)), str(value), fill='black', font=font)
        short_label = str(label)[:18]
        draw.text((x0, height - margin + 8), short_label, fill='black', font=font)
    output = FIGURES_DIR / filename
    image.save(output)
    print(f'Saved figure: {output.relative_to(PROJECT_ROOT)}')

def save_histogram(values, title, filename, transform='identity', bins=30):
    cleaned = pd.Series(values).dropna().astype(float)
    if transform == 'log1p':
        cleaned = cleaned.map(math.log1p)
    lower, upper = cleaned.min(), cleaned.max()
    if lower == upper:
        counts, labels = [len(cleaned)], [f'{lower:.2f}']
    else:
        step = (upper - lower) / bins
        counts = [0] * bins
        for value in cleaned:
            index = min(int((value - lower) / step), bins - 1)
            counts[index] += 1
        labels = [f'{lower + i * step:.1f}' for i in range(bins)]
    save_bar_chart(labels, counts, title, filename, color=(77, 175, 74))


## 2. Dataset dimensions

In [ ]:
print(f'master_tweets.csv: {tweets.shape[0]:,} rows x {tweets.shape[1]} columns')
print(f'network_edges.csv: {edges.shape[0]:,} rows x {edges.shape[1]} columns')

## 3. Schema and data types

In [ ]:
print('Tweet schema:')
print(tweets.dtypes.astype(str).to_string())
print('\nNetwork schema:')
print(edges.dtypes.astype(str).to_string())

## 4. Missing-value analysis

In [ ]:
tweet_missing = tweets.isna().sum()
edge_missing = edges.isna().sum()
print('Tweet missing values:\n' + tweet_missing.to_string())
print('\nNetwork missing values:\n' + edge_missing.to_string())

## 5. Duplicate analysis

In [ ]:
tweet_duplicate_rows = int(tweets.duplicated().sum())
edge_duplicate_rows = int(edges.duplicated().sum())
repeated_edge_pairs = int(edges.duplicated(subset=['Source', 'Target']).sum())
print(f'Exact duplicate tweet rows: {tweet_duplicate_rows:,}')
print(f'Exact duplicate edge rows: {edge_duplicate_rows:,}')
print(f'Repeated Source-Target pairs: {repeated_edge_pairs:,}')

## 6. Date validation

In [ ]:
# Parse independently rather than trusting the supplied calendar columns.
# This detects malformed timestamps and inconsistent Year/Month/Day metadata.
tweets['Date_parsed'] = pd.to_datetime(tweets['Date'], errors='coerce', utc=True)
invalid_dates = int(tweets['Date_parsed'].isna().sum())
calendar_mismatch = (
    (tweets['Date_parsed'].dt.year != tweets['Year']) |
    (tweets['Date_parsed'].dt.month != tweets['Month']) |
    (tweets['Date_parsed'].dt.day != tweets['Day'])
).fillna(True)
date_component_mismatches = int(calendar_mismatch.sum())
print(f'Invalid/unparseable dates: {invalid_dates:,}')
print(f'Year/Month/Day mismatches: {date_component_mismatches:,}')
print(f'Date range (UTC): {tweets.Date_parsed.min()} to {tweets.Date_parsed.max()}')

## 7. Target distribution

In [ ]:
# Labels are reported exactly as stored; no class meaning is inferred here.
target_counts = tweets['Target'].value_counts(dropna=False).sort_index()
print(target_counts.to_string())
save_bar_chart(target_counts.index.astype(str).tolist(), target_counts.astype(int).tolist(),
               'Target distribution (labels as stored)', 'target_distribution.png')

## 8. Unique-user analysis

In [ ]:
unique_users = int(tweets['User'].nunique())
print(f'Unique tweet users: {unique_users:,}')

## 9. Likes and Retweets descriptive statistics

In [ ]:
engagement_summary = tweets[['Likes', 'Retweets']].describe(percentiles=[0.25, 0.5, 0.75]).T
print(engagement_summary.to_string())
print('\nNegative Likes:', int((tweets['Likes'] < 0).sum()))
print('Negative Retweets:', int((tweets['Retweets'] < 0).sum()))
print('Non-integral Likes:', int((tweets['Likes'] % 1 != 0).sum()))
print('Non-integral Retweets:', int((tweets['Retweets'] % 1 != 0).sum()))

## 10. Engagement distribution visualizations

In [ ]:
# log1p makes the long-tailed engagement distributions interpretable without discarding outliers.
save_histogram(tweets['Likes'], 'Likes distribution (log1p scale)', 'likes_distribution.png', transform='log1p')
save_histogram(tweets['Retweets'], 'Retweets distribution (log1p scale)', 'retweets_distribution.png', transform='log1p')

## 11. Monthly tweet volume

In [ ]:
monthly_frame = tweets.dropna(subset=['Date_parsed']).copy()
# Remove timezone only for month bucketing; Date_parsed itself remains UTC for validation.
monthly_volume = monthly_frame.groupby(monthly_frame['Date_parsed'].dt.tz_localize(None).dt.to_period('M')).size()
print(monthly_volume.to_string())
save_bar_chart(monthly_volume.index.astype(str).tolist(), monthly_volume.astype(int).tolist(),
               'Monthly tweet volume', 'monthly_tweet_volume.png', color=(228, 26, 28))

## 12. Top users by tweet count

In [ ]:
top_users = tweets['User'].value_counts().head(15)
print(top_users.to_string())
save_bar_chart(top_users.index.tolist(), top_users.astype(int).tolist(),
               'Top 15 users by tweet count', 'top_users.png', color=(152, 78, 163))

## 13. Network edge statistics

In [ ]:
source_nodes = set(edges['Source'].astype(str))
target_nodes = set(edges['Target'].astype(str))
network_nodes = source_nodes | target_nodes
out_degree = edges['Source'].value_counts()
in_degree = edges['Target'].value_counts()
print(f'Edges: {len(edges):,}')
print(f'Unique source nodes: {len(source_nodes):,}')
print(f'Unique target nodes: {len(target_nodes):,}')
print(f'Unique network nodes: {len(network_nodes):,}')
print('\nWeight summary:\n' + edges['Weight'].describe().to_string())
degree_summary = {'Median out-degree': int(out_degree.median()), 'Max out-degree': int(out_degree.max()),
                  'Median in-degree': int(in_degree.median()), 'Max in-degree': int(in_degree.max())}
save_bar_chart(list(degree_summary), list(degree_summary.values()),
               'Network degree summary', 'network_degree_summary.png', color=(255, 127, 0))

## 14. Tweet-user/network-node overlap

In [ ]:
tweet_users = set(tweets['User'].astype(str))
overlapping_users = tweet_users & network_nodes
print(f'Overlapping users: {len(overlapping_users):,}')
print(f'Overlap as share of tweet users: {len(overlapping_users) / len(tweet_users):.2%}')
print(f'Overlap as share of network nodes: {len(overlapping_users) / len(network_nodes):.2%}')
print(f'Tweet-only users: {len(tweet_users - network_nodes):,}')
print(f'Network-only nodes: {len(network_nodes - tweet_users):,}')

## 15. Self-loop analysis

In [ ]:
self_loops = edges.loc[edges['Source'] == edges['Target']]
print(f'Self-loop edges: {len(self_loops):,} ({len(self_loops) / len(edges):.2%} of all edges)')
print(self_loops.head(10).to_string(index=False))

## 16. Repeated tweet-text analysis

In [ ]:
tweet_text_frequency = tweets['Tweet'].value_counts()
repeated_text_values = int((tweet_text_frequency > 1).sum())
rows_with_repeated_text = int(tweet_text_frequency[tweet_text_frequency > 1].sum())
additional_repeated_text_rows = int(rows_with_repeated_text - repeated_text_values)
print(f'Tweet-text values appearing more than once: {repeated_text_values:,}')
print(f'Rows represented by repeated tweet text: {rows_with_repeated_text:,}')
print(f'Additional repeated-text rows beyond the first occurrence: {additional_repeated_text_rows:,}')

## 17. Network identifier validation

In [ ]:
# This is a format check only. It does not prove account existence or infer identity.
# The pattern follows the conventional 1-15 ASCII letter/digit/underscore handle format.
identifier_pattern = re.compile(r'^[A-Za-z0-9_]{1,15}$')
def invalid_identifiers(values):
    unique_values = sorted(set(values.astype(str)))
    return [value for value in unique_values if not identifier_pattern.fullmatch(value)]

invalid_tweet_users = invalid_identifiers(tweets['User'])
invalid_sources = invalid_identifiers(edges['Source'])
invalid_targets = invalid_identifiers(edges['Target'])
print(f'Invalid tweet-user identifiers: {len(invalid_tweet_users):,}')
print(f'Invalid Source identifiers: {len(invalid_sources):,}')
print(f'Invalid Target identifiers: {len(invalid_targets):,}')
print('Target examples:', invalid_targets[:10])

## 18. Final data-quality summary

In [ ]:
quality_findings = [
    f'Missing tweet fields: {int(tweet_missing.sum()):,}; missing edge fields: {int(edge_missing.sum()):,}.',
    f'Exact duplicate tweet rows: {tweet_duplicate_rows:,}; exact duplicate edge rows: {edge_duplicate_rows:,}.',
    f'Unparseable dates: {invalid_dates:,}; date-component mismatches: {date_component_mismatches:,}.',
    f'Self-loop edges: {len(self_loops):,}.',
    f'Network Target identifiers failing the format check: {len(invalid_targets):,}.',
    'Likes and Retweets are stored as floats but all observed values are non-negative integers.',
    'Likes and Retweets are long-tailed; use robust summaries or transformations in later analysis.',
    'Target labels are reported as stored; their meaning remains intentionally unassigned.'
]
for finding in quality_findings:
    print('- ' + finding)

# Compact machine-readable summary for downstream review; it does not write a new dataset.
audit_summary = {
    'tweet_count': int(len(tweets)),
    'unique_users': unique_users,
    'network_edge_count': int(len(edges)),
    'network_node_count': int(len(network_nodes)),
    'overlapping_users': int(len(overlapping_users)),
    'target_counts': {str(label): int(count) for label, count in target_counts.items()},
    'date_range_utc': {'start': str(tweets.Date_parsed.min()), 'end': str(tweets.Date_parsed.max())},
    'missing_values': {'tweets': {key: int(value) for key, value in tweet_missing.items()},
                       'edges': {key: int(value) for key, value in edge_missing.items()}},
    'duplicate_rows': {'tweets': tweet_duplicate_rows, 'edges': edge_duplicate_rows}
}
print('\nProgrammatic audit summary:')
print(json.dumps(audit_summary, indent=2))